# Tires

Peak friction with load sensitivity and camber, and a simplified Magic Formula lateral curve
(`lapsim/subsystems/tires.py`):

    mu(Fz, gamma) = lambda_mu (mu0 + mu_fz (Fz - Fz0)) (1 - k_gamma (gamma - gamma_opt)^2)

The slope of the lateral curve at zero slip is the cornering stiffness (C_alpha / Fz times Fz); its peak is
mu Fz. The g-g diagram is the tire + differential envelope of the whole car (`Vehicle.traction_ggv`).

In [ ]:
%matplotlib inline
from nb import *

In [ ]:
CAR = lapsim.load_car("ct17_ic")
SL = [slider(CAR, "vehicle:tires.mu0", 1.0, 2.2, 0.01, "mu0 (peak mu at Fz0)"),
      slider(CAR, "vehicle:tires.mu_fz", -0.0012, 0.0, 0.00001, "mu_fz (load sensitivity)", 1, "1/N"),
      slider(CAR, "vehicle:tires.k_gamma", 0.0, 0.03, 0.0005, "k_gamma (camber loss)", 1, "1/deg^2"),
      slider(CAR, "vehicle:tires.gamma_opt_deg", -3, 1, 0.1, "best camber", 1, "deg"),
      slider(CAR, "vehicle:tires.cornering_stiffness_per_load", 8, 40, 0.5, "C_alpha / Fz", 1, "1/rad"),
      slider(CAR, "vehicle:tires.lambda_mu", 0.8, 1.2, 0.01, "grip scale (lambda_mu)")]
out, out_ggv = W.Output(), W.Output()


def draw(*_):
    with out:
        out.clear_output(wait=True)
        c = with_values(CAR, SL)
        tire = lapsim.subsystems.tires.Tire(c.vehicle.tires)
        fz = np.linspace(200, 1800, 50)
        al = np.radians(np.linspace(0, 15, 61))
        fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
        for g in (-2, -1, 0, 1):
            ax[0].plot(fz, tire.mu(fz, g), label=f"camber {g} deg")
        ax[0].set(xlabel="vertical load [N]", ylabel="peak mu", title="Peak friction"); ax[0].legend()
        for z in (400, 800, 1200):
            ax[1].plot(np.degrees(al), tire.fy(al, z, 0.0), label=f"Fz {z} N")
        ax[1].set(xlabel="slip angle [deg]", ylabel="Fy [N]", title="Lateral force"); ax[1].legend()
        fx = np.linspace(0, 1, 50) * tire.peak_fx(800.0)
        cap, _ = tire.lateral_capacity(800.0, 0.0, fx)
        ax[2].plot(fx, cap); ax[2].set(xlabel="Fx [N]", ylabel="Fy available [N]", title="Friction ellipse, Fz 800 N")
        plt.show()


def ggv():
    veh = with_values(CAR, SL).build()
    g = veh.traction_ggv()
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    for i in range(0, len(g.v), 6):
        ay = g.frac * g.ay_max[i] / 9.81
        ax.plot(np.r_[ay, ay[::-1]], np.r_[g.ax_acc[i], -g.ax_brk[i][::-1]] / 9.81, label=f"{3.6 * g.v[i]:.0f} km/h")
    ax.set(xlabel="ay [g]", ylabel="ax [g]", title="g-g diagram (tires + differential)"); ax.legend(fontsize=8)
    plt.show()


for s in SL:
    s.observe(draw, "value")
draw()
display(W.VBox([*SL, out, run_button("g-g diagram of the car", ggv, out_ggv), out_ggv]))